<a href="https://colab.research.google.com/github/wagmacaravan/scvi-scanvi-lung-integration/blob/main/scVI_scANVI_lung_integration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Batch integration and cell-type label transfer of lung scRNA-seq with scVI / scANVI

Trains a variational autoencoder (**scVI**) to integrate 32,472 cells across 16 batches
of the [scIB lung atlas benchmark](https://theislab.github.io/scib-reproducibility/),
then a semi-supervised model (**scANVI**) for cross-batch cell-type label transfer.
Integration quality is quantified against a PCA baseline with the `scib-metrics` suite,
and label transfer is evaluated on a held-out query set.

**Pipeline**
1. Load data and QC the raw counts
2. Train scVI and read the loss curve
3. Visualize integration (UMAP) and quantify it vs PCA
4. Train scANVI and re-benchmark three ways
5. Held-out label transfer: accuracy, per-type breakdown, confidence triage, confusion matrix

**Environment:** Google Colab, single T4 GPU. `scvi-tools` 1.5, `scib-metrics`.

**Key results:** scVI more than doubled batch-correction score over PCA (0.29 to 0.61) at ~1 pt
bio-conservation cost; scANVI recovered and exceeded bio-conservation (0.71) for best overall
integration. Held-out label transfer reached **91.4% accuracy**; the main systematic error was
Basal 2 → Basal 1 (shared lineage), with Secretory acting as a mild sink for airway epithelial types.


## 1. Setup and data loading

In [ ]:
# Install (Colab). scib-metrics is only needed for the benchmark sections.
!pip install --quiet scvi-tools scib-metrics

In [ ]:
import numpy as np
import scanpy as sc
import scvi
import torch

# Seed Python, NumPy, and PyTorch together for reproducibility.
scvi.settings.seed = 0

print("scvi-tools version:", scvi.__version__)
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

In [ ]:
# Download and load the preprocessed lung atlas (~628 MB; cached after first run).
adata = sc.read(
    "lung_atlas_preprocessed.h5ad",
    backup_url="https://exampledata.scverse.org/scvi-tools/lung_atlas_preprocessed.h5ad",
)
adata

## 2. QC the raw counts

scVI models **raw integer counts** with a negative-binomial likelihood. It accounts for
library size and overdispersion itself. Feeding it normalized/log/scaled data is the most
common scVI mistake, so we verify the counts are genuinely raw before training.

In [ ]:
# Confirm layers["counts"] holds raw integer counts (not normalized).
counts = adata.layers["counts"]
print("dtype:", counts.dtype)

sample = counts[:500].toarray()

# All values non-negative integers?
print("all integers?", np.array_equal(sample, np.round(sample)))
print("min:", sample.min(), " max:", sample.max())

# Normalized data sums to a constant per cell; raw counts do not.
row_sums = sample.sum(axis=1)
print("row sums (first 5):", row_sums[:5])
print("row sums all equal?", np.allclose(row_sums, row_sums[0]))

In [ ]:
# Counts are raw but stored as float32, which trips scVI's count-check heuristic.
# Cast to int (lossless here, since we just confirmed all values are whole numbers).
adata.layers["counts"] = adata.layers["counts"].astype(np.int64)
print("new dtype:", adata.layers["counts"].dtype)

In [ ]:
# Batch / cell-type landscape and confounding check.
# The groupby confirms each batch spans most cell types, so batch and biology are separable
print("=== batches ===")
print(adata.obs["batch"].value_counts())
print("\n=== cell types ===")
print(adata.obs["cell_type"].value_counts())
print("\n=== distinct cell types per batch ===")
print(adata.obs.groupby("batch", observed=True)["cell_type"].nunique())

## 3. Train scVI

Register the data (pointing scVI at the raw-count layer and the batch column), then train.
`early_stopping` halts when validation ELBO plateaus, so `max_epochs` is the ceiling.

In [ ]:
scvi.model.SCVI.setup_anndata(
    adata,
    layer="counts",
    batch_key="batch",
)

In [ ]:
# n_latent=30: dimensionality of the per-cell latent summary that replaces PCA.
model = scvi.model.SCVI(adata, n_latent=30)
model

In [ ]:
model.train(
    max_epochs=500,
    early_stopping=True,
    early_stopping_patience=15,
    check_val_every_n_epoch=1,
)

In [ ]:
# Train vs validation loss.
import matplotlib.pyplot as plt

history = model.history
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(history["elbo_train"], label="train")
ax.plot(history["elbo_validation"], label="validation")
ax.set_xlabel("epoch")
ax.set_ylabel("ELBO (loss; lower is better)")
ax.set_title("scVI training vs validation loss")
ax.legend()
plt.show()

### (Optional) Save / reload the trained model

On Colab, mount Drive first. The reliable reload pattern is to rebuild `adata` from source
(deterministic, since everything is seeded) and pass it explicitly to `load()`.

In [ ]:
# Save (optional). Requires Drive mounted: from google.colab import drive; drive.mount('/content/drive')
# model.save("/content/drive/MyDrive/scvi_lung_model", overwrite=True, save_anndata=True)

# Reload later:
# model = scvi.model.SCVI.load("/content/drive/MyDrive/scvi_lung_model", adata=adata)

## 4. Visualize and quantify integration

Extract the latent space, build a UMAP on it, and check the two panels:
batch should be **mixed** (correction worked) while cell types stay **separated** (biology preserved).

In [ ]:
# 30 batch-corrected numbers per cell — this replaces PCA downstream.
adata.obsm["X_scVI"] = model.get_latent_representation()
print(adata.obsm["X_scVI"].shape)

In [ ]:
# Neighbor graph + UMAP computed on the scVI latent space (not PCA).
sc.pp.neighbors(adata, use_rep="X_scVI")
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata, color=["batch", "cell_type"], frameon=False, ncols=2)

### Quantify vs a PCA baseline (scib-metrics)

The eyeball test isn't proof so `scib-metrics` scores two competing goals of batch correction
and bio-conservation, both of which trade off against each other. PCA is the uncorrected baseline.

In [ ]:
# Uncorrected PCA baseline embedding.
sc.tl.pca(adata)
print(adata.obsm["X_pca"].shape)

In [ ]:
from scib_metrics.benchmark import Benchmarker

bm = Benchmarker(
    adata,
    batch_key="batch",
    label_key="cell_type",
    embedding_obsm_keys=["X_pca", "X_scVI"],
    n_jobs=-1,
)
bm.benchmark()

In [ ]:
bm.plot_results_table(min_max_scale=False)

**Result:** scVI Total 0.64 vs PCA 0.52. Batch correction 0.61 vs 0.29 (>2×);
bio-conservation 0.66 vs 0.67 (PCA edges it out, because it never risks merging populations).
scVI removed batch variance at ~1 pt bio-conservation cost. Exact metric values vary by ~±0.01 across runs

## 5. scANVI: add label supervision

scANVI warm-starts from the trained scVI model and refines the latent space using the
`cell_type` labels, which should sharpen cell-type boundaries (raising bio-conservation).
Few epochs are needed because it refines rather than trains from scratch.

In [ ]:
scanvi_model = scvi.model.SCANVI.from_scvi_model(
    model,
    adata=adata,
    labels_key="cell_type",
    unlabeled_category="Unknown",
)
scanvi_model

In [ ]:
# n_samples_per_label=100 balances the sampling so rare types (e.g. Ionocytes, n=46)
# are not drowned out by Macrophages (n=7492).
scanvi_model.train(max_epochs=20, n_samples_per_label=100)

In [ ]:
adata.obsm["X_scANVI"] = scanvi_model.get_latent_representation()
print(adata.obsm["X_scANVI"].shape)

In [ ]:
# Re-benchmark three ways: PCA vs scVI vs scANVI.
bm = Benchmarker(
    adata,
    batch_key="batch",
    label_key="cell_type",
    embedding_obsm_keys=["X_pca", "X_scVI", "X_scANVI"],
    n_jobs=-1,
)
bm.benchmark()

In [ ]:
bm.plot_results_table(min_max_scale=False)

**Result:** monotonic ladder on Total — scANVI 0.66 > scVI 0.64 > PCA 0.52.
Bio-conservation rose to 0.71 (beating even PCA), concentrated in clustering-recovery metrics
(KMeans NMI 0.66 to 0.76), for a 3 pt batch-correction trade. Label supervision sharpened
cell-type structure.

## 6. Held-out label transfer

The real test of scANVI: predict labels for cells it never saw a label for. We hide the labels
on a random 20% **query** set (stashing the truth as an answer key), train scANVI on the
**reference** 80%, then grade its predictions on the query.

In [ ]:
# Stash true labels, pick a 20% query set, blank its labels to "Unknown".
adata.obs["true_label"] = adata.obs["cell_type"].astype(str)

rng = np.random.default_rng(0)
is_query = rng.random(adata.n_obs) < 0.20
adata.obs["is_query"] = is_query

adata.obs["labels_for_scanvi"] = adata.obs["cell_type"].astype(str)
adata.obs.loc[is_query, "labels_for_scanvi"] = "Unknown"

print("query cells:", is_query.sum(), "of", adata.n_obs)

In [ ]:
# Re-register with the partially-hidden labels, warm-start a fresh scANVI.
scvi.model.SCANVI.setup_anndata(
    adata,
    labels_key="labels_for_scanvi",
    unlabeled_category="Unknown",
    layer="counts",
    batch_key="batch",
)

scanvi_lt = scvi.model.SCANVI.from_scvi_model(
    model,
    adata=adata,
    labels_key="labels_for_scanvi",
    unlabeled_category="Unknown",
)

In [ ]:
scanvi_lt.train(max_epochs=20, n_samples_per_label=100)

In [ ]:
# Predict all cells; grade the query set against the stashed truth.
adata.obs["predicted_label"] = scanvi_lt.predict()

query = adata.obs[adata.obs["is_query"]].copy()
overall_acc = (query["predicted_label"] == query["true_label"]).mean()
print(f"Overall query accuracy: {overall_acc:.3f}")
print(f"Query cells graded: {len(query)}")

**Result:** 91.8% accuracy on 6,490 held-out cells.

In [ ]:
# Per-cell-type accuracy — the overall number hides the rare classes.
import pandas as pd

per_type = query.groupby("true_label").apply(
    lambda g: (g["predicted_label"] == g.name).mean(),
    include_groups=False,
)

counts = query["true_label"].value_counts()

summary = pd.DataFrame({"accuracy": per_type, "n_query": counts}).sort_values(
    "n_query", ascending=False
)
print(summary)

**Finding:** rarity is not difficulty as Ionocytes reached 92% with only ~34 reference cells (labeled correctly 11/12 times (92%) despite scANVI having only ~34 labeled Ionocytes in the reference to learn from), while the *lowest* accuracies were common types with close neighbors (Basal 2 at 84%).

## 7. Prediction confidence as a triage signal

On real unlabeled data there is no answer key, so we need to know which predictions to trust.
scANVI outputs a probability per cell type; the max probability is its confidence.
We test whether low confidence actually flags errors.

In [ ]:
# Full probability distribution per cell (rows sum to 1).
soft = scanvi_lt.predict(soft=True)
print(type(soft), soft.shape)
soft.head()

In [ ]:
# Confidence = max probability per cell. Compare on correct vs wrong query calls.
adata.obs["confidence"] = soft.max(axis=1).to_numpy()

query = adata.obs[adata.obs["is_query"]].copy()
query["correct"] = query["predicted_label"] == query["true_label"]

print("Mean confidence when CORRECT:", round(query.loc[query["correct"], "confidence"].mean(), 3))
print("Mean confidence when WRONG:  ", round(query.loc[~query["correct"], "confidence"].mean(), 3))

In [ ]:
# Does a confidence threshold usefully triage errors?
# precision = fraction of flagged cells that are truly wrong
# recall = fraction of all errors that fall below the threshold
for thresh in [0.70, 0.80, 0.90, 0.95, 0.99]:
    low_conf = query["confidence"] < thresh
    n_flagged = low_conf.sum()
    if n_flagged == 0:
        continue
    precision = (~query.loc[low_conf, "correct"]).mean()
    recall = (~query.loc[low_conf, "correct"]).sum() / (~query["correct"]).sum()
    print(f"thresh {thresh}: flag {n_flagged:>4} cells | "
          f"{precision:.0%} of flagged are errors | catches {recall:.0%} of all errors")

**Finding:** confidence carries real signal (correct calls avg 0.98, wrong 0.83) but the
distributions overlap with no clean threshold separating them, consistent with known neural-network
overconfidence (Guo et al., 2017). A "review the lowest-confidence ~10%" rule (thresh ≈ 0.90)
flags 576 cells, catches ~half the errors, and is ~45% precise. Useful for prioritizing manual
review and not a substitute for validation.

## 8. Confusion matrix

Which types get mistaken for which; testing whether errors follow biological structure.

In [ ]:
from sklearn.metrics import confusion_matrix

labels = sorted(query["true_label"].unique())
cm = confusion_matrix(query["true_label"], query["predicted_label"], labels=labels)

# Row-normalize: "of all true type X, what fraction went where" (handles class imbalance).
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)

ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=90); ax.set_yticklabels(labels)
ax.set_xlabel("predicted label"); ax.set_ylabel("true label")
ax.set_title("scANVI label transfer — query confusion matrix (row-normalized)")
fig.colorbar(im, ax=ax, label="fraction of true-type cells")

# Overlay the value in each cell.
for i in range(len(labels)):          # i = row index (true label)
    for j in range(len(labels)):      # j = column index (predicted label)
        val = cm_norm[i, j]
        if val < 0.005:               # skip near-zero cells to reduce clutter
            continue
        # white text on dark cells, so its always readable
        color = "white" if val > 0.5 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                color=color, fontsize=8)

plt.tight_layout()
plt.show()

**Finding:**
(1) A lineage-pair confusion: 11% of Basal 2 cells called Basal 1.
(2) Secretory acts as a mild sink, absorbing 2–8% of several airway epithelial types
(Ionocytes, Ciliated, Basal 2) which is consistent with secretory cells' intermediate position in
airway epithelial differentiation. A milder myeloid confusion (Macrophage/Dendritic, 3–6%) is
also present. Predicted alveolar (Type 1/Type 2) and neutrophil-subset confusions did **not**
appear as those are at 95–98%.

---

### References
- Lopez et al. (2018), *Deep generative modeling for single-cell transcriptomics*, Nature Methods.
- Xu et al. (2021), *Probabilistic harmonization and annotation of single-cell transcriptomics data with deep generative models* (scANVI), Molecular Systems Biology 17(1):e9620.
- Luecken et al. (2022), *Benchmarking atlas-level data integration in single-cell genomics* (scIB metrics), Nature Methods.
- Guo et al. (2017), *On Calibration of Modern Neural Networks*, ICML.
